In [1]:
# %pip install -qU requests pymupdf ipywidgets pandas pydantic
# %pip install -qU langchain langchain-community langchain-openai langchain-text-splitters pypdf chromadb


In [2]:
# Importações básicas
import os

# Loader de documentos PDF
from langchain_community.document_loaders import PyMuPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Banco vetorial
from langchain_community.vectorstores import Chroma

# LLM and Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Cadeia RAG
from langchain_core.prompts import ChatPromptTemplate

#

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

print("[✅]Imports loaded")

[✅]Imports loaded


In [3]:
# =================================================================
# 1. CONFIGURATION (Pydantic V2 Settings)
# =================================================================

import os
import json
from typing import Annotated, List, Optional, Literal, TypedDict

# Third-party: Pydantic
from pydantic import Field, SecretStr, ConfigDict, BaseModel,  field_validator
from pydantic_settings import BaseSettings

# Third-party: Utilities
from dotenv import load_dotenv

load_dotenv()

class ProjectConfig(BaseSettings):
    """Immutable project configuration with LLM + Embedding support via LM Studio."""
    model_config = ConfigDict(frozen=True, env_file=".env", extra="ignore")
    
    # LLM Configuration
    llm_base_url: str = Field(default="http://localhost:1234/v1")
    llm_model_name: str = Field(default="qwen/qwen3-4b-2507")
    llm_temperature: float = Field(default=0.0, ge=0.0, le=2.0)
    openai_api_key: SecretStr = Field(default=SecretStr("not-needed"))
    
    # Embedding Configuration (Nomic via LM Studio)
    embedding_base_url: str = Field(default="http://localhost:1234/v1")
    embedding_model_name: str = Field(default="nomic-embed-text-v1.5")

    def get_chat_model(self) -> ChatOpenAI:
        return ChatOpenAI(
            base_url=self.llm_base_url,
            model=self.llm_model_name,
            temperature=self.llm_temperature,
            api_key=self.openai_api_key.get_secret_value(),
            max_retries=2,
            timeout=120.0
        )

    def get_embedding_model(self) -> OpenAIEmbeddings:
        """Embeddings OpenAI-compatible via LM Studio."""
        return OpenAIEmbeddings(
            base_url=self.embedding_base_url,
            model=self.embedding_model_name,
            api_key=self.openai_api_key.get_secret_value(),
            check_embedding_ctx_length=False,  # Critical: Nomic uses 8192 tokens vs OpenAI's 2048
            tiktoken_enabled=False,            # Nomic doesn't use OpenAI's tokenizer
        )

settings = ProjectConfig()
model = settings.get_chat_model()
embedding_model = settings.get_embedding_model() 

test_embedding = embedding_model.embed_query("test")
embedding_dim = len(test_embedding)

print(f"[✅] LLM '{settings.llm_model_name}' ready at {settings.llm_base_url}")
print(f"[✅] Embedder '{settings.embedding_model_name}' configured at {settings.embedding_base_url}")
print(f"[✅] Embedding dimension: {embedding_dim} (expected: 768 for Nomic)")

[✅] LLM 'qwen/qwen3-4b-2507' ready at http://localhost:1234/v1
[✅] Embedder 'nomic-embed-text-v1.5' configured at http://localhost:1234/v1
[✅] Embedding dimension: 768 (expected: 768 for Nomic)


In [4]:
# %% [markdown]
# ### 1. Download do PDF via Stream (GitHub Raw)
# Carrega o documento diretamente da URL sem salvar no disco

# %%
import pymupdf
import requests

# URL do PDF no GitHub (raw)
PDF_URL = "https://raw.githubusercontent.com/alura-cursos/5564-langchain/main/Projeto1/regras_futebol.pdf"

# Baixar conteúdo
response = requests.get(PDF_URL)
response.raise_for_status()

# Validar que é realmente um PDF
if not response.content.startswith(b"%PDF"):
    raise ValueError("Conteúdo baixado não é um PDF válido!")

# Carregar documento na memória
doc = pymupdf.Document(stream=response.content, filetype="pdf")

print(f"✅ PDF carregado com sucesso!")
print(f"📄 Total de páginas: {doc.page_count}")
print(f"🔖 Metadados: {doc.metadata.get('creator', 'N/A')} | {doc.metadata.get('producer', 'N/A')}")

✅ PDF carregado com sucesso!
📄 Total de páginas: 232
🔖 Metadados: Adobe InDesign 18.3 (Windows) | 3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)


In [5]:
# %% [markdown]
# ### 2. Extração de Todas as Páginas para DataFrame Temporário
# Captura conteúdo bruto de cada página + métricas básicas

# %%
import pandas as pd

# Extrair todas as páginas
pages_data = []
for page_num in range(doc.page_count):
    page = doc[page_num]
    text_raw = page.get_text()
    
    pages_data.append({
        "page_number": page_num + 1,
        "content_raw": text_raw,
        "char_count": len(text_raw),
        "line_count": len(text_raw.split("\n"))
    })

doc.close()  # Fechar documento após extração

# Criar DataFrame temporário
df_temp = pd.DataFrame(pages_data)

# Adicionar conteúdo limpo (remover quebras excessivas)
df_temp["content_clean"] = df_temp["content_raw"].str.replace(r"\n+", " ", regex=True).str.strip()

print(f"✅ DataFrame temporário criado com {len(df_temp)} páginas")
print("\n📊 Amostra das primeiras 5 páginas:")
print(df_temp[["page_number", "char_count", "line_count", "content_clean"]].head())

✅ DataFrame temporário criado com 232 páginas

📊 Amostra das primeiras 5 páginas:
   page_number  char_count  line_count  \
0            1          55           7   
1            2           2           2   
2            3           2           2   
3            4         355           9   
4            5          26           5   

                                       content_clean  
0  1 Regras  do Jogo 23/24 Baixe o app das Regras...  
1                                                  2  
2                                                  3  
3  4 Entrada em vigor: 1º de julho de 2023 Esta p...  
4                          5 Regras  do Jogo 2023/24  


In [6]:
# %% [markdown]
# ### 3. Limpeza Inteligente: Remover Páginas Vazias/Irrelevantes
# Filtra páginas com apenas números de página, cabeçalhos repetitivos ou conteúdo mínimo

# %%
import re

def is_meaningful_page(text: str) -> bool:
    """
    Retorna True se a página tem conteúdo significativo.
    Critérios:
    - Não é apenas número de página
    - Tem pelo menos 20 caracteres significativos
    - Contém pelo menos uma palavra com 3+ letras
    - Não é apenas cabeçalho repetitivo ("Regras do Jogo de Futebol")
    """
    cleaned = re.sub(r"\s+", " ", text.strip())
    
    # Remover cabeçalhos repetitivos típicos de PDFs de regras
    headers = [
        r"^Regras do Jogo de Futebol$",
        r"^IFAB$",
        r"^International Football Association Board$",
    ]
    for header in headers:
        cleaned = re.sub(header, "", cleaned, flags=re.IGNORECASE | re.MULTILINE).strip()
    
    # Remover apenas números (com pontuação simples)
    if re.fullmatch(r"[\d\s\-\–—.,:;]*", cleaned):
        return False
    
    # Muito curto
    if len(cleaned) < 20:
        return False
    
    # Sem palavras reais (sequência de 3+ letras)
    if not re.search(r"[a-zA-Zà-úÀ-Ú]{3,}", cleaned):
        return False
    
    return True

# Aplicar filtro
df_temp["is_meaningful"] = df_temp["content_clean"].apply(is_meaningful_page)

# Criar DataFrame limpo
df_clean = df_temp[df_temp["is_meaningful"]].copy().reset_index(drop=True)

print(f"🧹 Limpeza concluída!")
print(f"   Total original: {len(df_temp)} páginas")
print(f"   Mantidas: {len(df_clean)} páginas ({len(df_clean)/len(df_temp)*100:.1f}%)")
print(f"   Removidas: {len(df_temp) - len(df_clean)} páginas")

# Mostrar páginas removidas (exemplo)
removed_pages = df_temp[~df_temp["is_meaningful"]]["page_number"].tolist()[:15]
print(f"\n🗑️ Exemplo de páginas removidas: {removed_pages}")

# Visualizar amostra do conteúdo limpo
print("\n✅ Amostra do conteúdo limpo (primeiras 3 páginas úteis):")
for idx, row in df_clean.head(3).iterrows():
    preview = row["content_clean"][:250].replace("\n", " ")
    print(f"\nPágina {row['page_number']} (chars: {row['char_count']}):")
    print(f"   {preview}...")

🧹 Limpeza concluída!
   Total original: 232 páginas
   Mantidas: 180 páginas (77.6%)
   Removidas: 52 páginas

🗑️ Exemplo de páginas removidas: [2, 3, 8, 9, 10, 13, 19, 27, 28, 30, 40, 43, 44, 52, 59]

✅ Amostra do conteúdo limpo (primeiras 3 páginas úteis):

Página 1 (chars: 55):
   1 Regras  do Jogo 23/24 Baixe o app das Regras do Jogo...

Página 4 (chars: 355):
   4 Entrada em vigor: 1º de julho de 2023 Esta publicação não pode ser reproduzida nem traduzida integral ou parcialmente de nenhuma maneira sem a  autorização de The International Football Association Board. The International Football Association Boar...

Página 5 (chars: 26):
   5 Regras  do Jogo 2023/24...


In [7]:
# %% [markdown]
# ### 4. Salvar Versão Oficial Limpa
# Exporta para CSV/Parquet pronto para uso com LangChain, embeddings e RAG

# %%
from pathlib import Path

# Criar diretório de saída
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Selecionar colunas relevantes para RAG
df_official = df_clean[[
    "page_number",
    "content_clean",
    "char_count"
]].rename(columns={"content_clean": "content"}).copy()

# Adicionar metadados úteis
df_official["source"] = "regras_futebol.pdf"
df_official["document_type"] = "fifa_laws_of_the_game"
df_official["total_original_pages"] = len(df_temp)
df_official["total_clean_pages"] = len(df_clean)

# Salvar em múltiplos formatos
csv_path = output_dir / "regras_futebol_clean.csv"
parquet_path = output_dir / "regras_futebol_clean.parquet"

df_official.to_csv(csv_path, index=False, encoding="utf-8")
df_official.to_parquet(parquet_path, index=False)

print(f"✅ Versão oficial salva com sucesso!")
print(f"   📁 CSV:  {csv_path.resolve()}")
print(f"   📁 Parquet: {parquet_path.resolve()}")
print(f"\n📊 Resumo final:")
print(f"   - Páginas úteis: {len(df_official)}")
print(f"   - Caracteres totais: {df_official['char_count'].sum():,}")
print(f"   - Média por página: {df_official['char_count'].mean():.0f} chars")

# Preview final
print("\n🔍 Preview do DataFrame oficial:")
print(df_official[["page_number", "char_count", "content"]].head(3))

✅ Versão oficial salva com sucesso!
   📁 CSV:  /home/yuri/Projetos/Langchain_2.0/data/processed/regras_futebol_clean.csv
   📁 Parquet: /home/yuri/Projetos/Langchain_2.0/data/processed/regras_futebol_clean.parquet

📊 Resumo final:
   - Páginas úteis: 180
   - Caracteres totais: 214,357
   - Média por página: 1191 chars

🔍 Preview do DataFrame oficial:
   page_number  char_count                                            content
0            1          55  1 Regras  do Jogo 23/24 Baixe o app das Regras...
1            4         355  4 Entrada em vigor: 1º de julho de 2023 Esta p...
2            5          26                          5 Regras  do Jogo 2023/24


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Configurar splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Prioriza quebras semânticas
)

# Preparar dados
texts = df_official['content'].tolist()
# ⚠️ REMOVER char_count do metadata dos chunks (é da página, não do chunk)
metadatas = [{"page_number": int(pn)} for pn in df_official['page_number']]

# Criar chunks
chunks = text_splitter.create_documents(texts, metadatas=metadatas)

# Enriquecer metadata com informações do chunk
for i, chunk in enumerate(chunks):
    chunk.metadata.update({
        "chunk_index": i,  # ID global do chunk
        "chunk_size": len(chunk.page_content),  # Tamanho REAL do chunk
        "source": "regras_futebol.pdf"
    })

# Relatório
print(f"✅ Chunking concluído!")
print(f"   Páginas originais: {len(df_official)}")
print(f"   Chunks gerados: {len(chunks)}")
print(f"   Média de chunks por página: {len(chunks) / len(df_official):.1f}")
print(f"   Tamanho médio dos chunks: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

# Amostra dos primeiros chunks
print("\n🔍 Amostra dos primeiros 3 chunks:")
for i, chunk in enumerate(chunks[:3]):
    preview = chunk.page_content[:150].replace('\n', ' ')
    print(f"\nChunk #{chunk.metadata['chunk_index']} | Página {chunk.metadata['page_number']} | {chunk.metadata['chunk_size']} chars")
    print(f"   {preview}...")

✅ Chunking concluído!
   Páginas originais: 180
   Chunks gerados: 647
   Média de chunks por página: 3.6
   Tamanho médio dos chunks: 346 chars

🔍 Amostra dos primeiros 3 chunks:

Chunk #0 | Página 1 | 54 chars
   1 Regras  do Jogo 23/24 Baixe o app das Regras do Jogo...

Chunk #1 | Página 4 | 353 chars
   4 Entrada em vigor: 1º de julho de 2023 Esta publicação não pode ser reproduzida nem traduzida integral ou parcialmente de nenhuma maneira sem a  auto...

Chunk #2 | Página 5 | 25 chars
   5 Regras  do Jogo 2023/24...


In [9]:
# Cria o banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_regras_futebol"
)

print(f"✅ Vectorstore criado com {len(chunks)} chunks!")

✅ Vectorstore criado com 647 chunks!


In [10]:
# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ========================================
# 1. FORMATAÇÃO ENRIQUECIDA DO CONTEXTO
# ========================================
def format_docs(docs):
    """
    Formata documentos com metadados para melhor rastreabilidade.
    """
    formatted = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page_number", "?")
        content = doc.page_content.strip()
        # Limitar a 800 caracteres por trecho para foco
        if len(content) > 800:
            content = content[:797] + "..."
        formatted.append(f"[Trecho {i} | Página {page}]\n{content}")
    return "\n\n".join(formatted)

# ========================================
# 2. PROMPT SYSTEM ROBUSTO (anti-alucinação)
# ========================================
prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente especialista nas Regras Oficiais do Futebol (IFAB).

📌 INSTRUÇÕES ESTRITAS:
1. RESPONDA EXCLUSIVAMENTE com base no contexto fornecido abaixo.
2. Se o contexto NÃO contiver informações suficientes, responda: "Não encontrei informações sobre isso nas regras oficiais."
3. Cite sempre a regra ou seção relevante quando possível (ex: "Regra 12").
4. Seja objetivo e técnico — evite opiniões ou exemplos não presentes no contexto.
5. Respostas devem ser claras e diretas, com no máximo 3 parágrafos.

📚 CONTEXTO DAS REGRAS OFICIAIS:
{contexto}"""),
    ("human", "{pergunta}")
])

# ========================================
# 3. PIPELINE LCEL OTIMIZADO (simples e robusto)
# ========================================
rag_chain = (
    {
        "contexto": retriever | RunnableLambda(format_docs),
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

# ========================================
# 4. TESTE COM PERGUNTA REAL
# ========================================
pergunta_teste = "Um jogador pode usar a mão para marcar um gol?"

print("❓ Pergunta:", pergunta_teste)
print("\n💬 Resposta:")
resposta = rag_chain.invoke(pergunta_teste)
print(resposta)

# ========================================
# 5. BÔNUS: Streaming nativo (já funciona!)
# ========================================
print("\n" + "="*50)
print("✨ Demonstração de Streaming (UX melhorada):")
print("="*50)
for chunk in rag_chain.stream("O que acontece se a bola tocar o braço acidentalmente?"):
    print(chunk, end="", flush=True)
print("\n✅ Streaming concluído!")

❓ Pergunta: Um jogador pode usar a mão para marcar um gol?

💬 Resposta:
Não, um jogador não pode usar a mão para marcar um gol. De acordo com a Regra 12 das Regras Oficiais do Futebol (IFAB), tocar a bola com a mão ou o braço constitui uma infração se for feito deliberadamente, por exemplo, ao deslocar a mão ou o braço na direção da bola, ou quando esses membros ampliarem o corpo do jogador de maneira antinatural.  

No entanto, é importante destacar que, embora a mão não possa ser usada para marcar um gol, há exceções específicas, como o caso do goleiro, que está autorizado a usar as mãos dentro da área de seu gol.  

Portanto, qualquer jogador fora do goleiro que tocar a bola com a mão ou o braço com o intuito de marcar um gol está cometendo uma infração e será penalizado.

✨ Demonstração de Streaming (UX melhorada):
Se a bola tocar o braço acidentalmente, isso constitui uma infração, mesmo que seja acidental. De acordo com a Regra 12, o jogador que coloca a mão ou o braço na posição

In [29]:
from langchain_core.runnables import RunnablePassthrough

pergunta_teste = "Um jogador pode usar a mão para marcar um gol?"

# 1. Recuperar documentos fonte (para exibição educacional)
docs_fonte = retriever.invoke(pergunta_teste)

print("="*60)
print("🔍 TRECHOS UTILIZADOS COMO CONTEXTO (para aprendizagem)")
print("="*60)
for i, doc in enumerate(docs_fonte, start=1):
    page = doc.metadata.get('page_number', 'N/A')
    content_preview = doc.page_content[:400].replace('\n', ' ')
    if len(doc.page_content) > 400:
        content_preview += "..."
    
    print(f"\n📄 Trecho {i} | Página {page}")
    print("-" * 60)
    print(content_preview)
print("\n" + "="*60)

# 2. Gerar resposta final (pipeline LCEL)
resposta = rag_chain.invoke(pergunta_teste)

print("\n💬 RESPOSTA DO ASSISTENTE:")
print("="*60)
print(resposta)
print("="*60)

🔍 TRECHOS UTILIZADOS COMO CONTEXTO (para aprendizagem)

📄 Trecho 1 | Página 102
------------------------------------------------------------
. Nem  todos os contatos da mão ou do braço de um jogador com a bola constituem  uma infração. No entanto, cometerá uma infração o jogador que:  • tocar na bola com sua mão ou seu braço deliberadamente; por exemplo,  deslocando a mão ou o braço na direção da bola; • tocar na bola com sua mão ou seu braço quando estes ampliarem o corpo do  jogador de maneira antinatural

📄 Trecho 2 | Página 102
------------------------------------------------------------
. Nem  todos os contatos da mão ou do braço de um jogador com a bola constituem  uma infração. No entanto, cometerá uma infração o jogador que:  • tocar na bola com sua mão ou seu braço deliberadamente; por exemplo,  deslocando a mão ou o braço na direção da bola; • tocar na bola com sua mão ou seu braço quando estes ampliarem o corpo do  jogador de maneira antinatural

📄 Trecho 3 | Página 102
